# Using Regex in Python

In this example we're going to use regular expressions to extract data from free-text. This is quite a common use of regular expressions, particularly when webscraping lists *etc.* Download the *student_details.txt* file from Brightspace and drop it into the same folder as this notebook to follow along.

Before we can get our hands dirty with regular expressions we need to read the text into python. Our first step is to read this file into a variable. Regular Expressions are part of the Python standard library but we need to import them if we want to use them in our scripts. The regular expression module is named **re**.

In [ ]:
import re

text = None

with open("./student_details.txt") as f:
    text = f.read()
    
text

We've successfully read in the text file and we've stored in a variable called **text**. Notice the **\n** characters in the text above. This is a character encoding meaning *new line*, if we opened this in a text editor, all of the \n characters would be replaced by a line break.

Our next step is to define the goal. First we need to work out what are the different values which can be extracted from this text. Every line seems to contain four pieces of information, *name*, *student number*, *course code*, *date of birth*. We'll start small and work our way up. Our first task is to extract the name from each line (first and last). Remember, to extract data we need to use *capturing groups*.

In [ ]:
pattern = "(\w+) (\w+)"
regex = re.compile(pattern, re.MULTILINE) # this is a multiline file (it contains linebreaks or \n characters)
matches = regex.findall(text)
matches

Not a bad start, we've successfully matched *John* and *Doe*. Unfortunately, we've matched pretty much every other word, too. Each line should be a new record, so we can use the $ character to match the end of the line. We're going to capture our first name, then our last name, and then match (but not capture) everything up to the end of the line

In [ ]:
pattern = "(\w+) (\w+).*$"
regex = re.compile(pattern, re.MULTILINE) # this is a multiline file (it contains linebreaks or \n characters)
matches = regex.findall(text)
matches

We have to write these same four lines of code every time we want to check an updated pattern. Let's define a function to make our lives easier. 

In [ ]:
def test_pattern(pattern):
    regex = re.compile(pattern, re.MULTILINE)
    matches = regex.findall(text)
    return matches

test_pattern("(\w+) (\w+).*$")

That seems fine, but Richard Boyd Barrett lost his double-barrel. How do we know whether a student has two or three names? If we look closely at the text we can see it follows a regular pattern

```
<student name> has student number...
```

If we include the *has student number* in our match then we'll be able to tell whether the student has 2 or 3 names. We may have multiple surnames, so we'll add the space character to our surname character class to allow Boyd Barrett to match. This second capturing group will match any combination of letters spaces and apostrophes until it finds the text **has student number**.

In [ ]:
test_pattern("(\w+) ([\w' ]+) has student number.*$")

The next part of our entry is the student number itself. We want to capture this. How do we define a student number?

A student number is a letter, (capital C or capital D) followed by 8 digits, Let's add this to our pattern in a capturing group

In [ ]:
test_pattern("(\w+) ([\w' ]+) has student number ([CD][\d]{8}).*$")

Now we've got the student number, let's extract the course enrollment

```
John Doe has student number D12345665, is enrolled on TU953 and was born on 01/02/1990
```

We know that the student number will be followed by **, is enrolled on** . We can match that text and then capture our course code. The course code is going to be the letters *TU* followed by 3 numbers

In [ ]:
test_pattern("(\w+) ([\w' ]+) has student number ([CD][\d]{8}), is enrolled on (TU\d\d\d).*$")

Have you noticed anything about the output? We seem to have lost Bojan.

```
Bojan Bozick has student number C87654321, is enrolled on TU44 and was born on 08/08/2002
```

On closer inspection, we can see that Bojan's course only has 2 digits. We can fix using repetition

In [ ]:
test_pattern("(\w+) ([\w' ]+) has student number ([CD][\d]{8}), is enrolled on (TU[\d]{2,3}).*$")

Finally, we want to match the date of birth. The date of birth is in the format dd/mm/yyyy. The date of birth should be the last part of the string, so we should put our dollar right after our final capturing group. The dot-star will match the *and was born on* part of the text for us

In [ ]:
test_pattern("(\w+) ([\w' ]+) has student number ([CD][\d]{8}), is enrolled on (TU[\d]{2,3}).* (\d{2}/\d{2}/\d{4})$")

We've done it. Now we can pull out the variables we matched using capturing groups. I'll re-write the code from test_pattern below just for clarity

In [ ]:
pattern = "(\w+) ([\w' ]+) has student number ([CD][\d]{8}), is enrolled on (TU[\d]{2,3}).* (\d{2}/\d{2}/\d{4})$"
regex = re.compile(pattern, re.MULTILINE)
matches = regex.findall(text)

for match in matches:
    first, last, std_no, course, dob = match
    print("first: " + first)
    print("last: " + last)
    print("std_no " + std_no)
    print("course " + course)
    print("dob " + dob)
    print()

That's not really very neat. If we use a format string it'll be easier to print it nicely

In [ ]:
for match in matches:
    first, last, std_no, course, dob = match
    print(f"first: {first}, last: {last}, std_no: {std_no}, course: {course}, dob: {dob}")

## Your Turn

Download the file *ftse_100_salaries.txt*. This file contains the name, company and salary of the top 20 highest paid CEOs of ftse 100 companies. Some of the lines are credits for photographs and should be ignored (if they don't match your regex they'll be ignored).

Your task is extract the relevant information from each entry in this file and output a string for each in the following format

```
<name> is <rank>th on the list, working for <company> and making <amount>
```

*Bonus: Output the correct suffix for 2nd and 1st (this isn't really regex specific but can be done through regular Python*

Antonio Horta Osorio is 20th in the list, working for Lloyds Group  and making £5.5 million
Stuart Gulliver is 19th in the list, working for HSBC  and making £5.7 million
Xavier Rolet is 18th in the list, working for London Stock Exchange Group  and making £5.7 million
Simon Borrows is 17th in the list, working for 3i Group  and making £5.8 milliion
Richard Cousins is 16th in the list, working for Compass Group  and making £5.8 million
Peter Harrison is 15th in the list, working for Schroders  and making £6.3 million
Peter Crook is 14th in the list, working for Provident Financial  and making £6.3 million
Paul Polman is 13th in the list, working for Unilever  and making £6.7 million
Andrew Witty is 12th in the list, working for GlaxoSmithKline  and making £6.8 million
Mike Wells is 11th in the list, working for Prudential  and making £6.9 million
Ben Van Beurden is 10th in the list, working for Royal Dutch Shell  and making £6.9 million
Flemming Ornskov is 9th in the list, working for 